# Decision Tree Regression for Abnormal Returns Prediction

This notebook implements a decision tree regression model to predict abnormal returns based on sentiment scores and tweet volume.

In [1]:
# Import required packages
import pandas as pd
import numpy as np
import os
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import r2_score, mean_squared_error

c:\Users\skazempour\AppData\Local\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
# Directories
PROJECT_DIR = r"C:/Users/skazempour/Dropbox/Projects/42 - Machine learning from the crowd/"

CODE_DIR = os.path.join(PROJECT_DIR, "Code")
DATA_DIR = os.path.join(PROJECT_DIR, "Data")
FIGURES_DIR = os.path.join(PROJECT_DIR, "Figures")
TABLES_DIR = os.path.join(PROJECT_DIR, "Tables")

# File names
INPUT_DATA = os.path.join(DATA_DIR, "merged_master.pkl")

In [3]:
data = pd.read_pickle(INPUT_DATA)

In [4]:
# Prepare features and target
# Target variable
TARGET = 'f_cumret1'

# Feature engineering
# data['log_n_tweets'] = np.log10(1 + data['n_tweets'])

# Select features
FEATURES = ['net_sentiment', 'log_volume']

# Remove missing values
model_data = data[[TARGET] + FEATURES].dropna()

print(f"Sample size: {len(model_data):,}")
print(f"Target: {TARGET}")
print(f"Features: {FEATURES}")

Sample size: 16,743,676
Target: f_cumret1
Features: ['net_sentiment', 'log_volume']


# In-Sample Decision Tree Regression

In [5]:
# Prepare X and y
X = model_data[FEATURES]
y = model_data[TARGET]

# Fit decision tree regression model (in-sample)
dt_model = DecisionTreeRegressor(random_state=42)
dt_model.fit(X, y)

# Make predictions
y_pred = dt_model.predict(X)

# Evaluate performance
r2 = r2_score(y, y_pred)
mse = mean_squared_error(y, y_pred)
rmse = np.sqrt(mse)

print("In-Sample Decision Tree Regression Results")
print("=" * 50)
print(f"R-squared: {r2:.6f}")
print(f"RMSE: {rmse:.6f}")
print(f"MSE: {mse:.6f}")
print(f"\nTree depth: {dt_model.get_depth()}")
print(f"Number of leaves: {dt_model.get_n_leaves()}")
print(f"\nFeature importances:")
for feature, importance in zip(FEATURES, dt_model.feature_importances_):
    print(f"  {feature}: {importance:.6f}")

In-Sample Decision Tree Regression Results
R-squared: 0.023049
RMSE: 0.046550
MSE: 0.002167

Tree depth: 72
Number of leaves: 57825

Feature importances:
  net_sentiment: 0.599468
  log_volume: 0.400532


# Out-of-Sample Predictions

In [6]:
# Out-of-sample predictions with multiple window types
# Parameters
TRAIN_END_DATE = '2011-12-31'
WINDOW_252 = 252  # One trading year
WINDOW_21 = 21    # One trading month

# Ensure date column is datetime
model_data['date'] = pd.to_datetime(data.loc[model_data.index, 'date'])

# Sort by date
model_data = model_data.sort_values('date')

# Get unique dates
unique_dates = model_data['date'].unique()
unique_dates = pd.DatetimeIndex(unique_dates).sort_values()

# Find the starting prediction date (first date after training window)
train_end = pd.to_datetime(TRAIN_END_DATE)
oos_dates = unique_dates[unique_dates > train_end]

print(f"Training window: {unique_dates[0].strftime('%Y-%m-%d')} to {TRAIN_END_DATE}")
print(f"OOS prediction period: {oos_dates[0].strftime('%Y-%m-%d')} to {oos_dates[-1].strftime('%Y-%m-%d')}")
print(f"Number of OOS dates: {len(oos_dates):,}")

Training window: 2010-01-04 to 2011-12-31
OOS prediction period: 2012-01-03 to 2024-12-30
Number of OOS dates: 3,269


## Moderate Tree (max_depth=10, min_samples_leaf=50)

In [13]:
# Method 2: Moderate Tree (max_depth=10, min_samples_leaf=50)
print("=" * 70)
print("METHOD 2: MODERATE TREE (max_depth=10, min_samples_leaf=50)")
print("=" * 70)

# Initialize storage for predictions
predictions_moderate_exp = []
predictions_moderate_252 = []
predictions_moderate_21 = []

# Loop through OOS dates
for i, pred_date in enumerate(oos_dates):
    # Test data: observations on pred_date
    test_mask = model_data['date'] == pred_date
    X_test = model_data.loc[test_mask, FEATURES]
    
    if len(X_test) == 0:
        continue
    
    test_indices = model_data.index[test_mask]
    
    # 1. Expanding window
    train_mask_exp = model_data['date'] < pred_date
    X_train_exp = model_data.loc[train_mask_exp, FEATURES]
    y_train_exp = model_data.loc[train_mask_exp, TARGET]
    
    if len(X_train_exp) > 0:
        dt_moderate = DecisionTreeRegressor(max_depth=10, min_samples_leaf=50, random_state=42)
        dt_moderate.fit(X_train_exp, y_train_exp)
        y_pred_exp = dt_moderate.predict(X_test)
        
        for idx, pred in zip(test_indices, y_pred_exp):
            predictions_moderate_exp.append({
                'date': pred_date,
                'index': idx,
                'pred_expanding': pred
            })
    
    # 2. Rolling 252-day window
    date_idx = unique_dates.get_loc(pred_date)
    if date_idx >= WINDOW_252:
        start_date_252 = unique_dates[date_idx - WINDOW_252]
        train_mask_252 = (model_data['date'] >= start_date_252) & (model_data['date'] < pred_date)
        X_train_252 = model_data.loc[train_mask_252, FEATURES]
        y_train_252 = model_data.loc[train_mask_252, TARGET]
        
        if len(X_train_252) > 0:
            dt_moderate = DecisionTreeRegressor(max_depth=10, min_samples_leaf=50, random_state=42)
            dt_moderate.fit(X_train_252, y_train_252)
            y_pred_252 = dt_moderate.predict(X_test)
            
            for idx, pred in zip(test_indices, y_pred_252):
                predictions_moderate_252.append({
                    'date': pred_date,
                    'index': idx,
                    'pred_rolling_252': pred
                })
    
    # 3. Rolling 21-day window
    if date_idx >= WINDOW_21:
        start_date_21 = unique_dates[date_idx - WINDOW_21]
        train_mask_21 = (model_data['date'] >= start_date_21) & (model_data['date'] < pred_date)
        X_train_21 = model_data.loc[train_mask_21, FEATURES]
        y_train_21 = model_data.loc[train_mask_21, TARGET]
        
        if len(X_train_21) > 0:
            dt_moderate = DecisionTreeRegressor(max_depth=10, min_samples_leaf=50, random_state=42)
            dt_moderate.fit(X_train_21, y_train_21)
            y_pred_21 = dt_moderate.predict(X_test)
            
            for idx, pred in zip(test_indices, y_pred_21):
                predictions_moderate_21.append({
                    'date': pred_date,
                    'index': idx,
                    'pred_rolling_21': pred
                })
    
    # Progress update
    if (i + 1) % 100 == 0:
        print(f"Processed {i + 1}/{len(oos_dates)} dates ({100 * (i + 1) / len(oos_dates):.1f}%)")

print(f"\nCompleted.")
print(f"Expanding window predictions: {len(predictions_moderate_exp):,}")
print(f"Rolling 252-day predictions: {len(predictions_moderate_252):,}")
print(f"Rolling 21-day predictions: {len(predictions_moderate_21):,}")

METHOD 2: MODERATE TREE (max_depth=10, min_samples_leaf=50)
Processed 100/2948 dates (3.4%)
Processed 100/2948 dates (3.4%)
Processed 200/2948 dates (6.8%)
Processed 200/2948 dates (6.8%)
Processed 300/2948 dates (10.2%)
Processed 300/2948 dates (10.2%)
Processed 400/2948 dates (13.6%)
Processed 400/2948 dates (13.6%)
Processed 500/2948 dates (17.0%)
Processed 500/2948 dates (17.0%)
Processed 600/2948 dates (20.4%)
Processed 600/2948 dates (20.4%)
Processed 700/2948 dates (23.7%)
Processed 700/2948 dates (23.7%)
Processed 800/2948 dates (27.1%)
Processed 800/2948 dates (27.1%)
Processed 900/2948 dates (30.5%)
Processed 900/2948 dates (30.5%)
Processed 1000/2948 dates (33.9%)
Processed 1000/2948 dates (33.9%)
Processed 1100/2948 dates (37.3%)
Processed 1100/2948 dates (37.3%)
Processed 1200/2948 dates (40.7%)
Processed 1200/2948 dates (40.7%)
Processed 1300/2948 dates (44.1%)
Processed 1300/2948 dates (44.1%)
Processed 1400/2948 dates (47.5%)
Processed 1400/2948 dates (47.5%)
Processed 

In [14]:
# Merge and save Method 2 predictions
df_moderate_exp = pd.DataFrame(predictions_moderate_exp)
df_moderate_252 = pd.DataFrame(predictions_moderate_252)
df_moderate_21 = pd.DataFrame(predictions_moderate_21)

predictions_moderate = df_moderate_exp.merge(
    df_moderate_252,
    on=['date', 'index'],
    how='outer'
).merge(
    df_moderate_21,
    on=['date', 'index'],
    how='outer'
)

# Add symbol information
predictions_moderate = predictions_moderate.merge(
    data[['symbol']].reset_index(),
    left_on='index',
    right_on='index',
    how='left'
)

# Reorder and sort
predictions_moderate = predictions_moderate[['date', 'symbol', 'index', 'pred_expanding', 'pred_rolling_252', 'pred_rolling_21']]
predictions_moderate = predictions_moderate.sort_values(['date', 'symbol']).reset_index(drop=True)

# Save predictions
OUTPUT_FILE_MODERATE = os.path.join(OUTPUT_DATA_DIR, "predictions_decision_tree_moderate.pkl")
predictions_moderate.to_pickle(OUTPUT_FILE_MODERATE)

print("\nMethod 2 - Moderate Tree Summary")
print("=" * 50)
print(f"Total observations: {len(predictions_moderate):,}")
print(f"Non-null predictions by window type:")
print(f"  Expanding: {predictions_moderate['pred_expanding'].notna().sum():,}")
print(f"  Rolling 252-day: {predictions_moderate['pred_rolling_252'].notna().sum():,}")
print(f"  Rolling 21-day: {predictions_moderate['pred_rolling_21'].notna().sum():,}")
print(f"\nSaved to: {OUTPUT_FILE_MODERATE}")
print(f"File size: {os.path.getsize(OUTPUT_FILE_MODERATE) / (1024**2):.2f} MB")


Method 2 - Moderate Tree Summary
Total observations: 3,001,267
Non-null predictions by window type:
  Expanding: 3,001,267
  Rolling 252-day: 3,001,267
  Rolling 21-day: 3,001,267

Saved to: C:/Users/skazempour/Dropbox/Projects/42 - Machine learning from the crowd/Data\predictions_decision_tree_moderate.pkl
File size: 128.85 MB
